# Modelo de precios de vivienda: versión definitiva (E7)

Estima el **precio de mercado pedido** de un apartamento, apartaestudio o casa en cualquier ciudad de
Colombia, con un **rango p10–p90** y un nivel de **confianza**. Vive en `src/modelo_precio.py`
(`entrenar`, `ModeloPrecio.predecir`, `evaluar`). Este notebook no entrena: lee el modelo guardado y su ficha.

> Es una estimación del precio de mercado pedido, **no un avalúo** (Ley 1673 de 2013).

## Cómo se llegó aquí

| Notebook | Pregunta | Resultado |
|---|---|---|
| [modelo_precio_e7](modelo_precio_e7.ipynb) | ¿Un modelo le gana a la mediana del barrio y a los comparables de la alerta? | Sí: 11,9% frente a 17,6% y 14,5% |
| [comparacion_modelos_e7](comparacion_modelos_e7.ipynb) | ¿Qué algoritmo? (9 probados) | *Boosting*; XGBoost y LightGBM ~0,3 puntos mejor que el primero |
| [variables_e7](variables_e7.ipynb) | ¿Qué información nueva ayuda? | Mercado alrededor, servicios OSM y texto: 11,6% → 10,7% |
| [fuentes_oficiales_e7](fuentes_oficiales_e7.ipynb) | ¿Censo DANE e IDESC (Cali)? | No mejoran el precio (sirven para describir la zona) |

## Qué quedó definido

| | Decisión | Por qué |
|---|---|---|
| Algoritmo | **LightGBM** solo | El ensamble con XGBoost ganaba ~0,1 puntos al doble de costo |
| Objetivo | log del precio por m² pedido | El precio estimado es `exp(predicción) × área` |
| Datos | Avisos de vivienda **activos** del barrido nacional, limpios de errores evidentes | `limpiar()`: precio/m² 0,8–35 M, sin extremos frente a su ciudad y tipo, sin duplicados |
| Variables | Ficha del inmueble, ubicación (coordenadas + barrio/zona/comuna/ciudad codificados), 40 amenidades, **mercado alrededor** (mediana del m² de los 10 y 30 vecinos del mismo tipo), **servicios de OSM** (17) y **texto** (14 palabras + puntaje TF-IDF) | Ablación de `variables_e7` |
| Fuera | Anunciante, días publicado, palabras del vendedor ("urgente", "negociable"), números del texto | El modelo aprendería "quién vende barato" o leería el precio escrito y escondería las oportunidades |
| Rango | Modelos de cuantiles 10 y 90 + calibración *conformal* | Contiene el 80% de los precios de verdad |
| Confianza | Alta / Media / Baja por el ancho del rango (tercios) | Separa bien el error esperado |
| Para la alerta | `entrenar(..., publicados_antes=fecha)` | Así califica avisos que **no** vio al entrenar |

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE))
from src import modelo_precio as mp

pct = lambda x: f"{x:.1%}".replace(".", ",")
mil = lambda x: f"{x:,.0f}".replace(",", ".")
modelo = mp.ModeloPrecio.cargar()
ficha = json.loads(mp.FICHA.read_text(encoding="utf-8"))
print(f"{ficha['modelo']}\nDatos al {ficha['fecha_datos']} · {mil(modelo.n_entrenamiento)} avisos de entrenamiento · "
      f"{len(modelo.columnas)} variables")

LightGBM · base + mercado alrededor + servicios OSM + texto
Datos al 2026-10-26 · 122.387 avisos de entrenamiento · 100 variables


## Ficha: qué tan bien estima

Medida con el **mismo camino de predicción que producción**, en avisos que el modelo no vio:
- **por inmueble:** 20% de los inmuebles fuera del entrenamiento (las copias de un mismo inmueble van juntas);
- **temporal:** entrena con lo publicado hasta 4 semanas antes y predice lo publicado después (lo más
  parecido a la alerta).

In [2]:
FORMATO = {"MdAPE": pct, "±10%": pct, "±20%": pct, "avisos": mil, "cobertura_p10_p90": pct, "ancho_mediano": pct}
general = pd.DataFrame({"Por inmueble": ficha["por_inmueble"],
                        f"Temporal (publicados desde el {ficha['temporal']['publicados_desde']})":
                            {k: v for k, v in ficha["temporal"].items() if k != "publicados_desde"}}).T
general.style.format(FORMATO)

,MdAPE,±10%,±20%,avisos,cobertura_p10_p90,ancho_mediano
Por inmueble,"10,9%","46,8%","74,7%",24.531,"78,8%","57,7%"
Temporal (publicados desde el 2026-09-28),"10,1%","49,7%","77,6%",16.649,"80,3%","53,3%"


In [3]:
display(pd.DataFrame(ficha["por_tipo"]).T.style.format(FORMATO).set_caption("Por tipo"))
display(pd.DataFrame(ficha["por_ciudad"]).T.style.format(FORMATO).set_caption("Por ciudad (las 10 con más avisos)"))
pd.DataFrame(ficha["por_confianza"]).T.loc[["Alta", "Media", "Baja"]].style.format(FORMATO).set_caption("Por nivel de confianza")

,MdAPE,±10%,±20%,avisos
Apartaestudio,"9,6%","51,8%","81,5%",826
Apartamento,"9,3%","52,8%","81,4%",15.513
Casa,"15,2%","34,9%","61,3%",8.192


,MdAPE,±10%,±20%,avisos
Bogotá,"9,7%","51,1%","78,8%",5.953
Medellín,"11,3%","44,8%","73,6%",2.931
Cali,"12,0%","43,2%","73,9%",1.741
Barranquilla,"12,4%","43,7%","70,0%",1.325
Envigado,"9,5%","51,7%","80,3%",1.140
Cartagena,"13,5%","38,0%","65,3%",906
Manizales,"11,2%","46,2%","73,4%",779
Pereira,"12,0%","42,4%","71,9%",705
Santa marta,"11,7%","44,1%","71,8%",585
Sabaneta,"7,9%","58,1%","88,4%",585


,MdAPE,±10%,±20%,avisos,cobertura_p10_p90
Alta,"7,5%","62,3%","90,0%",8.341,"80,9%"
Media,"10,8%","47,2%","77,5%",7.984,"79,5%"
Baja,"17,2%","30,7%","56,4%",8.206,"76,1%"


**Cómo leerla:**
- La mitad de las estimaciones queda a menos de ~11% del precio pedido, y tres de cada cuatro a menos de 20%.
- El rango p10–p90 contiene el precio en el ~80% de los casos, como se pidió. Es ancho (el extremo alto es
  ~55% más que el bajo), así que el punto y el rango van juntos.
- La **confianza** sirve: con "Alta", 9 de cada 10 quedan a menos de 20%; con "Baja", poco más de la mitad.
- **Casas** y ciudades costeras son lo más difícil (el área de las casas mezcla lote y construcción: E4).

## Cómo se usa

```bash
python -m src.modelo_precio entrenar                 # evalúa (ficha), entrena con todo y guarda (~6 min)
python -m src.modelo_precio entrenar --sin-evaluar   # solo entrena (~2 min)
python -m src.modelo_precio entrenar --publicados-antes 2026-10-26   # para calificar lo posterior
python -m src.modelo_precio predecir --ciudad Cali   # → reports/modelo/cali_<fecha>.csv
```

```python
from src import modelo_precio as mp
modelo = mp.ModeloPrecio.cargar()            # data/modelos/precio_vivienda.joblib (~42 MB, git lo ignora)
estimados = modelo.predecir(avisos)          # pm2_estimado, precio_estimado, precio_p10, precio_p90, confianza
```

`avisos` es un DataFrame con las columnas de `estado/avisos` (las del barrido). Si los avisos estaban en el
entrenamiento, la estimación se parece demasiado a su propio precio: para la alerta hay que entrenar con
`publicados_antes`.

## Pendiente para producción

1. Entrenamiento **semanal** en el CI y modelo + ficha en R2 (`--subir`).
2. **Alerta en paralelo:** "X% bajo el modelo" junto al descuento por comparables, unas semanas, antes de
   decidir cuál manda.
3. **Informe de captación** (producto 1): rango del modelo + comparables + descripción de la zona (IDESC/DANE).